In [1]:
import os
import sys
import time
import json
import random
import pickle
from pathlib import Path
from datetime import datetime
from typing import List, Tuple

# Standard data science libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings

# Machine learning libraries
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster

# PyTorch and Transformers
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoModel, AutoTokenizer

# ==============================================================================
# 1. REPRODUCIBILITY
# ==============================================================================
def setup_reproducibility(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

setup_reproducibility(42)

# ==============================================================================
# 2. MODELS & DATASETS
# ==============================================================================
class BALMProjectionHead(nn.Module):
    def __init__(self, embedding_size, projected_size, projected_dropout):
        super().__init__()
        self.protein_projection = nn.Linear(embedding_size, projected_size)
        self.proteina_projection = nn.Linear(embedding_size, projected_size)
        self.dropout = nn.Dropout(projected_dropout)
        self.loss_fn = nn.MSELoss()

    def forward(self, protein_emb, proteina_emb, labels=None):
        p1 = F.normalize(self.protein_projection(self.dropout(protein_emb)), p=2, dim=1)
        p2 = F.normalize(self.proteina_projection(self.dropout(proteina_emb)), p=2, dim=1)
        cosine_sim = torch.clamp(F.cosine_similarity(p1, p2), -0.9999, 0.9999)
        output = {"cosine_similarity": cosine_sim}
        if labels is not None:
            output["loss"] = self.loss_fn(cosine_sim, labels)
        return output

class BALMForFrozen(nn.Module):
    def __init__(self, embedding_size, projected_size, projected_dropout):
        super().__init__()
        self.projection_head = BALMProjectionHead(embedding_size, projected_size, projected_dropout)

    def forward(self, batch):
        out = self.projection_head(batch["protein_embedding"], batch["proteina_embedding"], batch.get("labels"))
        return {
            "cosine_similarity": out["cosine_similarity"],
            "original_pkds": batch["original_pkds"],
            "pdb_groups": batch["pdb_groups"],
            "subgroups": batch["subgroups"],
            "source_dataset": batch["source_dataset"],
            "loss": out.get("loss")
        }

class ProteinPairEmbeddingDataset(Dataset):
    def __init__(self, dataframe, pkd_bounds, embedding_map):
        self.data = dataframe.reset_index(drop=True)
        self.pkd_lower, self.pkd_upper = pkd_bounds
        self.pkd_range = self.pkd_upper - self.pkd_lower
        self.embedding_map = embedding_map

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        original_pkd = float(row['Y'])
        scaled_label = ((original_pkd - self.pkd_lower) / self.pkd_range) * 2 - 1
        return {
            "protein_embedding": self.embedding_map[str(row["Target"])],
            "proteina_embedding": self.embedding_map[str(row["proteina"])],
            "labels": torch.tensor(scaled_label, dtype=torch.float32),
            "original_pkds": torch.tensor(original_pkd, dtype=torch.float32),
            "pdb_groups": str(row["PDB"]),
            "subgroups": str(row["Subgroup"]),
            "source_dataset": str(row["SourceDataSet"])
        }

def collate_fn_embeddings(batch):
    return {
        "protein_embedding": torch.stack([item['protein_embedding'] for item in batch]),
        "proteina_embedding": torch.stack([item['proteina_embedding'] for item in batch]),
        "labels": torch.stack([item['labels'] for item in batch]),
        "original_pkds": torch.stack([item['original_pkds'] for item in batch]),
        "pdb_groups": [item['pdb_groups'] for item in batch],
        "subgroups": [item['subgroups'] for item in batch],
        "source_dataset": [item['source_dataset'] for item in batch]
    }

# ==============================================================================
# 3. HELPERS
# ==============================================================================
def compute_kmer_similarity_matrix(sequences, k=3):
    n = len(sequences)
    sim_matrix = np.zeros((n, n))
    kmers = []
    for s in sequences:
        if len(s) < k: kmers.append({s})
        else: kmers.append({s[i:i+k] for i in range(len(s) - k + 1)})
    for i in tqdm(range(n), desc="Computing similarities"):
        for j in range(i, n):
            if i == j: sim_matrix[i, j] = 1.0
            else:
                inter = len(kmers[i] & kmers[j])
                union = len(kmers[i] | kmers[j])
                sim = inter / union if union > 0 else 0.0
                sim_matrix[i, j] = sim_matrix[j, i] = sim
    return sim_matrix

def cluster_sequences_by_similarity(similarity_matrix, threshold=0.3):
    """Uses scipy for deterministic clustering"""
    distance_matrix = 1.0 - similarity_matrix
    distance_threshold = 1.0 - threshold
    
    print(f"Using distance threshold: {distance_threshold}")
    
    condensed_dist = squareform(distance_matrix, checks=False)
    Z = linkage(condensed_dist, method='average')
    clusters = fcluster(Z, t=distance_threshold, criterion='distance')
    clusters = clusters - 1
    
    print(f"Created {len(set(clusters))} clusters (deterministic)")
    
    return clusters

def evaluate_model(model, data_loader, device, pkd_bounds):
    model.eval()
    all_labels, all_preds, all_pdb, all_sub, all_src = [], [], [], [], []
    pkd_lower, pkd_range = pkd_bounds[0], pkd_bounds[1] - pkd_bounds[0]
    with torch.no_grad():
        for batch in data_loader:
            batch_gpu = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            outputs = model(batch_gpu)
            preds_pkd = ((outputs['cosine_similarity'].cpu() + 1) / 2) * pkd_range + pkd_lower
            all_labels.extend(batch['original_pkds'].numpy())
            all_preds.extend(preds_pkd.numpy())
            all_pdb.extend(batch['pdb_groups'])
            all_sub.extend(batch['subgroups'])
            all_src.extend(batch['source_dataset'])
    y_t, y_p = np.array(all_labels), np.array(all_preds)
    metrics = {'rmse': np.sqrt(mean_squared_error(y_t, y_p)), 'pearson': pearsonr(y_t, y_p)[0], 'spearman': spearmanr(y_t, y_p)[0]}
    return metrics, y_t, y_p, np.array(all_pdb), np.array(all_sub), np.array(all_src)

# ==============================================================================
# 4. MAIN CROSS-VALIDATION
# ==============================================================================
def run_cv_frozen():
    # --- Data Loading ---
    df_raw = pd.read_csv(r"D:\BALM_Fineclone\BALM-PPI\scripts\notebooks\PPB_Affinity_Sequences_Final (version 1).csv")
    df = df_raw[['Target', 'proteina', 'Y', 'PDB', 'Subgroup', 'Source Data Set']].copy()
    df.rename(columns={'Source Data Set': 'SourceDataSet'}, inplace=True)
    df.dropna(subset=['Target', 'proteina', 'Y', 'PDB'], inplace=True)
    df['Y'] = pd.to_numeric(df['Y'], errors='coerce')
    df.dropna(subset=['Y'], inplace=True)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    GLOBAL_BOUNDS = (df['Y'].min(), df['Y'].max())
    RESULTS_DIR = "cv_results_frozen_synced"
    os.makedirs(RESULTS_DIR, exist_ok=True)

    # --- Step 1: Extraction (SORTED for reproducibility) ---
    print("\n🧬 Step 1: Extracting unique sequences...")
    all_sequences_set = set()
    sequence_to_records = {}
    for idx, row in df.iterrows():
        for col in ['Target', 'proteina']:
            seq = str(row[col]).strip()
            if seq:
                all_sequences_set.add(seq)
                if seq not in sequence_to_records:
                    sequence_to_records[seq] = []
                sequence_to_records[seq].append(idx)
    
    # CRITICAL: Sort for reproducibility
    unique_sequences = sorted(list(all_sequences_set))
    print(f"Found {len(unique_sequences)} unique protein sequences (sorted).")

    # --- Step 2: Pre-compute Embeddings ---
    # FIXED: Correct cache path
    cache_path = r"esm2_embeddings_cache_new.pkl"
    
    if os.path.exists(cache_path):
        print("Loading embeddings from cache...")
        with open(cache_path, 'rb') as f: 
            embedding_map = pickle.load(f)
    else:
        print("Generating new embeddings...")
        extractor_model = AutoModel.from_pretrained("facebook/esm2_t33_650M_UR50D", torch_dtype=torch.float16).to(device).eval()
        tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
        embedding_map = {}
        
        for i in tqdm(range(0, len(unique_sequences), 4), desc="Generating Embeddings"):
            batch_original = unique_sequences[i:i+4]
            batch_for_model = [s.replace('|', f"{tokenizer.cls_token}{tokenizer.cls_token}") for s in batch_original]
            
            inputs = tokenizer(batch_for_model, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
            with torch.no_grad():
                out = extractor_model(**inputs).last_hidden_state
                mask = inputs['attention_mask'].unsqueeze(-1).expand(out.size()).float()
                mean_emb = torch.sum(out * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
                
                for original_seq, emb in zip(batch_original, mean_emb): 
                    embedding_map[original_seq] = emb.cpu()
        
        with open(cache_path, 'wb') as f: 
            pickle.dump(embedding_map, f)
        del extractor_model

    # --- Step 3: Similarity & Clustering ---
    sim_matrix = compute_kmer_similarity_matrix(unique_sequences, k=3)
    clusters = cluster_sequences_by_similarity(sim_matrix, threshold=0.3)
    
    cluster_to_records = {}
    for seq_idx, cluster_id in enumerate(clusters):
        seq = unique_sequences[seq_idx]
        if cluster_id not in cluster_to_records:
            cluster_to_records[cluster_id] = []
        cluster_to_records[cluster_id].extend(sequence_to_records[seq])
    
    for cluster_id in cluster_to_records:
        cluster_to_records[cluster_id] = list(set(cluster_to_records[cluster_id]))

    # --- Step 4: K-Fold with Preview ---
    cluster_ids = list(cluster_to_records.keys())
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    # NEW: Print fold preview
    print(f"\nCreated {len(set(clusters))} clusters")
    print(f"   Step 4: Creating 5-fold splits...")
    
    kf_preview = KFold(n_splits=5, shuffle=True, random_state=42)
    for preview_fold, (train_cluster_idx, test_cluster_idx) in enumerate(kf_preview.split(cluster_ids)):
        preview_fold_num = preview_fold + 1
        
        train_clusters_preview = set([cluster_ids[i] for i in train_cluster_idx])
        test_clusters_preview = set([cluster_ids[i] for i in test_cluster_idx])
        
        sequence_to_group_preview = {}
        for seq_idx, cluster_id in enumerate(clusters):
            seq = unique_sequences[seq_idx]
            sequence_to_group_preview[seq] = 'train' if cluster_id in train_clusters_preview else 'test'
        
        train_count, test_count = 0, 0
        for idx, row in df.iterrows():
            seq_target = str(row['Target']).strip()
            seq_proteina = str(row['proteina']).strip()
            
            if sequence_to_group_preview.get(seq_target) == 'test' or sequence_to_group_preview.get(seq_proteina) == 'test':
                test_count += 1
            else:
                train_count += 1
        
        print(f"   Fold {preview_fold_num}: Train={train_count}, Test={test_count}")
    
    print(f"\n🔄 Beginning actual training across 5 folds...")
    
    fold_results, all_y_t, all_y_p = [], [], []

    for fold, (train_idx, test_idx) in enumerate(kf.split(cluster_ids)):
        fold_num = fold + 1
        print(f"\n===== FOLD {fold_num}/5 =====")
        
        train_clusters = set([cluster_ids[i] for i in train_idx])
        test_clusters = set([cluster_ids[i] for i in test_idx])

        sequence_to_group = {}
        for seq_idx, cluster_id in enumerate(clusters):
            seq = unique_sequences[seq_idx]
            sequence_to_group[seq] = 'train' if cluster_id in train_clusters else 'test'

        # FIXED: Add .strip() for consistent lookup
        train_indices, test_indices = [], []
        for idx, row in df.iterrows():
            seq_target = str(row['Target']).strip()
            seq_proteina = str(row['proteina']).strip()
            
            if sequence_to_group.get(seq_target) == 'test' or sequence_to_group.get(seq_proteina) == 'test':
                test_indices.append(idx)
            else:
                train_indices.append(idx)

        train_df, test_df = df.loc[train_indices], df.loc[test_indices]
        print(f"  Train: {len(train_df)} | Test: {len(test_df)}")
        print(f"  Using Global Scaling Bounds: {GLOBAL_BOUNDS}")
        
        train_loader = DataLoader(ProteinPairEmbeddingDataset(train_df, GLOBAL_BOUNDS, embedding_map), 
                                  batch_size=8, shuffle=True, collate_fn=collate_fn_embeddings)
        test_loader = DataLoader(ProteinPairEmbeddingDataset(test_df, GLOBAL_BOUNDS, embedding_map), 
                                 batch_size=16, shuffle=False, collate_fn=collate_fn_embeddings)

        model = BALMForFrozen(1280, 256, 0.1).to(device)
        optimizer = AdamW(model.parameters(), lr=1e-4)
        
        best_val_rmse, patience_counter = float('inf'), 0
        best_model_path = f"{RESULTS_DIR}/best_fold_{fold_num}.pth"
        
        for epoch in range(30):
            model.train()
            train_loss_sum = 0
            for batch in train_loader:
                optimizer.zero_grad()
                batch_gpu = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
                loss = model(batch_gpu)['loss']
                loss.backward()
                optimizer.step()
                train_loss_sum += loss.item()
            
            avg_train_loss = train_loss_sum / len(train_loader)
            val_metrics, _, _, _, _, _ = evaluate_model(model, test_loader, device, GLOBAL_BOUNDS)
            current_val_rmse = val_metrics['rmse']
            
            print(f"Epoch {epoch+1}/30 | Train Loss: {avg_train_loss:.4f} | Val RMSE: {current_val_rmse:.4f} | Val Pearson: {val_metrics['pearson']:.4f}")
            
            if current_val_rmse < best_val_rmse:
                best_val_rmse = current_val_rmse
                patience_counter = 0
                torch.save(model.state_dict(), best_model_path)
                print(f"   -> New best model saved with Val RMSE: {best_val_rmse:.4f}")
            else:
                patience_counter += 1
            
            if patience_counter >= 15:
                print(f"   ⏳ Early stopping at epoch {epoch+1}. Val RMSE has not improved for 15 epochs.")
                break

        model.load_state_dict(torch.load(best_model_path))
        metrics, yt, yp, pdb, sub, src = evaluate_model(model, test_loader, device, GLOBAL_BOUNDS)
        
        print(f"Fold {fold_num} Final Test Metrics:")
        for key, value in metrics.items():
            print(f"  - {key.upper()}: {value:.4f}")
        
        fold_results.append(metrics)
        all_y_t.extend(yt)
        all_y_p.extend(yp)
        
        pred_df = pd.DataFrame({
            'label': yt,
            'prediction': yp,
            'residual': yt - yp,
            'abs_residual': np.abs(yt - yp),
            'PDB': pdb,
            'Subgroup': sub,
            'Source Data Set': src
        })
        pred_df.to_csv(f"{RESULTS_DIR}/fold_{fold_num}_predictions.csv", index=False)

    print("\n✅ CV Results Summary (Frozen Synced):")
    res_df = pd.DataFrame(fold_results)
    summary_df = pd.DataFrame({'Mean': res_df.mean(), 'Std Dev': res_df.std()})
    print(summary_df)
    
    summary_df.to_csv(f"{RESULTS_DIR}/cv_summary_metrics.csv")
    print(f"Summary saved to {RESULTS_DIR}/cv_summary_metrics.csv")

if __name__ == "__main__":
    run_cv_frozen()


🧬 Step 1: Extracting unique sequences...
Found 11598 unique protein sequences (sorted).
Generating new embeddings...


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Computing similarities: 100%|██████████| 11598/11598 [09:38<00:00, 20.04it/s] 


Using distance threshold: 0.7
Created 2678 clusters (deterministic)

Created 2678 clusters
   Step 4: Creating 5-fold splits...
   Fold 1: Train=6516, Test=5503
   Fold 2: Train=7503, Test=4516
   Fold 3: Train=7340, Test=4679
   Fold 4: Train=7331, Test=4688
   Fold 5: Train=9397, Test=2622

🔄 Beginning actual training across 5 folds...

===== FOLD 1/5 =====
  Train: 6516 | Test: 5503
  Using Global Scaling Bounds: (1.3187587626244128, 15.69897000433602)
Epoch 1/30 | Train Loss: 0.0478 | Val RMSE: 1.7131 | Val Pearson: 0.6552
   -> New best model saved with Val RMSE: 1.7131
Epoch 2/30 | Train Loss: 0.0363 | Val RMSE: 1.7653 | Val Pearson: 0.6153
Epoch 3/30 | Train Loss: 0.0338 | Val RMSE: 1.7662 | Val Pearson: 0.6092
Epoch 4/30 | Train Loss: 0.0315 | Val RMSE: 1.7615 | Val Pearson: 0.6311
Epoch 5/30 | Train Loss: 0.0297 | Val RMSE: 1.7263 | Val Pearson: 0.6617
Epoch 6/30 | Train Loss: 0.0288 | Val RMSE: 1.7068 | Val Pearson: 0.6330
   -> New best model saved with Val RMSE: 1.7068
Epoc

C:\Users\hs494\AppData\Local\Temp\ipykernel_54080\1675006234.py:356: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path))


Fold 1 Final Test Metrics:
  - RMSE: 1.7068
  - PEARSON: 0.6330
  - SPEARMAN: 0.6286

===== FOLD 2/5 =====
  Train: 7503 | Test: 4516
  Using Global Scaling Bounds: (1.3187587626244128, 15.69897000433602)
Epoch 1/30 | Train Loss: 0.0472 | Val RMSE: 1.9613 | Val Pearson: 0.3983
   -> New best model saved with Val RMSE: 1.9613
Epoch 2/30 | Train Loss: 0.0391 | Val RMSE: 2.0237 | Val Pearson: 0.3333
Epoch 3/30 | Train Loss: 0.0366 | Val RMSE: 1.9944 | Val Pearson: 0.3534
Epoch 4/30 | Train Loss: 0.0345 | Val RMSE: 1.9123 | Val Pearson: 0.3955
   -> New best model saved with Val RMSE: 1.9123
Epoch 5/30 | Train Loss: 0.0324 | Val RMSE: 1.9634 | Val Pearson: 0.3575
Epoch 6/30 | Train Loss: 0.0311 | Val RMSE: 1.9892 | Val Pearson: 0.3837
Epoch 7/30 | Train Loss: 0.0299 | Val RMSE: 1.9555 | Val Pearson: 0.4158
Epoch 8/30 | Train Loss: 0.0286 | Val RMSE: 2.1427 | Val Pearson: 0.3429
Epoch 9/30 | Train Loss: 0.0278 | Val RMSE: 1.9043 | Val Pearson: 0.4304
   -> New best model saved with Val RMSE

...

Fold 3 Final Test Metrics:
  - RMSE: 1.9411
  - PEARSON: 0.5518
  - SPEARMAN: 0.5725

===== FOLD 4/5 =====
  Train: 7331 | Test: 4688
  Using Global Scaling Bounds: (1.3187587626244128, 15.69897000433602)
Epoch 1/30 | Train Loss: 0.0513 | Val RMSE: 1.5401 | Val Pearson: 0.7006
   -> New best model saved with Val RMSE: 1.5401
Epoch 2/30 | Train Loss: 0.0419 | Val RMSE: 1.5383 | Val Pearson: 0.7183
   -> New best model saved with Val RMSE: 1.5383
Epoch 3/30 | Train Loss: 0.0377 | Val RMSE: 1.4938 | Val Pearson: 0.7105
   -> New best model saved with Val RMSE: 1.4938
Epoch 4/30 | Train Loss: 0.0358 | Val RMSE: 1.4879 | Val Pearson: 0.7136
   -> New best model saved with Val RMSE: 1.4879
Epoch 5/30 | Train Loss: 0.0340 | Val RMSE: 1.5019 | Val Pearson: 0.7126
Epoch 6/30 | Train Loss: 0.0325 | Val RMSE: 1.5394 | Val Pearson: 0.7056
Epoch 7/30 | Train Loss: 0.0315 | Val RMSE: 1.5497 | Val Pearson: 0.6998
Epoch 8/30 | Train Loss: 0.0300 | Val RMSE: 1.5831 | Val Pearson: 0.6983
Epoch 9/30 | Tr

C:\Users\hs494\AppData\Local\Temp\ipykernel_54080\1675006234.py:356: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path))


Fold 4 Final Test Metrics:
  - RMSE: 1.4879
  - PEARSON: 0.7136
  - SPEARMAN: 0.6920

===== FOLD 5/5 =====
  Train: 9397 | Test: 2622
  Using Global Scaling Bounds: (1.3187587626244128, 15.69897000433602)
Epoch 1/30 | Train Loss: 0.0479 | Val RMSE: 1.5806 | Val Pearson: 0.5839
   -> New best model saved with Val RMSE: 1.5806
Epoch 2/30 | Train Loss: 0.0389 | Val RMSE: 1.6527 | Val Pearson: 0.5688
Epoch 3/30 | Train Loss: 0.0363 | Val RMSE: 1.6014 | Val Pearson: 0.5862
Epoch 4/30 | Train Loss: 0.0347 | Val RMSE: 1.5679 | Val Pearson: 0.5989
   -> New best model saved with Val RMSE: 1.5679
Epoch 5/30 | Train Loss: 0.0332 | Val RMSE: 1.6440 | Val Pearson: 0.5656
Epoch 6/30 | Train Loss: 0.0319 | Val RMSE: 1.6433 | Val Pearson: 0.5493
Epoch 7/30 | Train Loss: 0.0305 | Val RMSE: 1.6381 | Val Pearson: 0.5638
Epoch 8/30 | Train Loss: 0.0297 | Val RMSE: 1.6288 | Val Pearson: 0.5679
Epoch 9/30 | Train Loss: 0.0286 | Val RMSE: 1.6811 | Val Pearson: 0.5451
Epoch 10/30 | Train Loss: 0.0278 | Val R

C:\Users\hs494\AppData\Local\Temp\ipykernel_54080\1675006234.py:356: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path))


Fold 5 Final Test Metrics:
  - RMSE: 1.5679
  - PEARSON: 0.5989
  - SPEARMAN: 0.5539

✅ CV Results Summary (Frozen Synced):
              Mean   Std Dev
rmse      1.717107  0.195047
pearson   0.582738  0.110177
spearman  0.566101  0.115444
Summary saved to cv_results_frozen_synced/cv_summary_metrics.csv
